# Week 3: pgvector RAG Demo

This notebook demonstrates the complete RAG pipeline with pgvector backend support:

1. Load documents and create chunks with page metadata
2. Generate embeddings (384-dimensional)
3. Store chunks in pgvector (or in-memory for testing)
4. Retrieve similar chunks using semantic search
5. Show citations and retrieval quality
6. Compare pgvector vs in-memory performance

## Setup: Import Libraries

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../../../'))

import numpy as np
import json
from typing import List, Dict
import time

# RAG modules
from rag.embedder import load_embedding_model, embed_texts, embed_query
from rag.chunker import create_chunks
from rag.vector_store import VectorStore
from rag.document_loader import DocumentLoader, pages_to_chunks
from rag.answer_generator import build_rag_prompt, generate_simple_answer

print(" All imports successful")

## Step 1: Create Sample Documents with Page Metadata

Simulating Duy's document_pages.jsonl format

In [ ]:
# Sample documents that simulate Duy's PDF extraction
sample_pages = [
    {
        "document_id": "doc_001",
        "file_name": "company_policy.pdf",
        "page_number": 1,
        "text": """Company Refund Policy
        
        Refunds are available within 30 days of purchase for most items. 
        To initiate a refund, customers must provide the original receipt or order confirmation. 
        Items must be in their original condition with all packaging intact. 
        Electronics must not show any signs of use. 
        Once approved, refunds are processed within 5-7 business days.
        """,
        "character_count": 400,
        "is_empty": False
    },
    {
        "document_id": "doc_001",
        "file_name": "company_policy.pdf",
        "page_number": 2,
        "text": """Returns Process
        
        For online purchases: Print the return label and drop off at any authorized location. 
        For in-store purchases: Bring items and receipt to customer service desk. 
        Return shipping is free for items exceeding $50 in value. 
        Items under $50 may have return shipping costs deducted from refund. 
        Special orders and clearance items are final sale and cannot be returned.
        """,
        "character_count": 380,
        "is_empty": False
    },
    {
        "document_id": "doc_001",
        "file_name": "company_policy.pdf",
        "page_number": 3,
        "text": """Warranty and Guarantees
        
        All products come with a 1-year manufacturer's warranty covering defects. 
        Warranty does not cover normal wear and tear or customer-caused damage. 
        Extended warranty options available for 2 or 5 years at checkout. 
        Warranty claims require proof of purchase and must be filed within 30 days of noticing defect. 
        Free repairs or replacement available during warranty period.
        """,
        "character_count": 420,
        "is_empty": False
    }
]

# Validate pages
stats = DocumentLoader.validate_pages(sample_pages)
print(f"\n Sample Document Statistics:")
print(f"   Total pages: {stats['total_pages']}")
print(f"   Non-empty pages: {stats['non_empty_pages']}")
print(f"   Total characters: {stats['total_characters']}")
print(f"   Unique documents: {stats['unique_documents']}")
print(f"   Valid: {stats['is_valid']}")

## Step 2: Convert Pages to Chunks with Page Metadata

In [ ]:
# Convert pages to chunks
chunks = pages_to_chunks(sample_pages, chunk_size=150, overlap=20)

print(f"\n️  Chunking Results:")
print(f"   Created {len(chunks)} chunks")
print(f"\n   Sample chunk structure:")
if chunks:
    sample_chunk = chunks[0]
    print(f"   - chunk_id: {sample_chunk['chunk_id']}")
    print(f"   - document_id: {sample_chunk['document_id']}")
    print(f"   - page_number: {sample_chunk['metadata']['page_number']}")
    print(f"   - text preview: {sample_chunk['chunk_text'][:80]}...")
    print(f"   - metadata: {sample_chunk['metadata']}")

# Show all chunk IDs
print(f"\n   Chunk IDs:")
for chunk in chunks:
    print(f"   - {chunk['chunk_id']} (page {chunk['metadata']['page_number']})")

## Step 3: Load Embedding Model and Generate Embeddings

In [ ]:
# Load embedding model
print(" Loading embedding model (all-MiniLM-L6-v2)...")
model = load_embedding_model()

# Extract chunk texts
chunk_texts = [chunk["chunk_text"] for chunk in chunks]

# Generate embeddings
print(f" Generating embeddings for {len(chunk_texts)} chunks...")
embeddings = embed_texts(chunk_texts, model)

print(f"\n Embeddings generated:")
print(f"   Shape: {embeddings.shape}")
print(f"   Embedding dimension: {embeddings.shape[1]}")
print(f"   Data type: {embeddings.dtype}")
print(f"   Sample embedding (first 10 values): {embeddings[0][:10]}")

## Step 4: Store Chunks in Vector Store (In-Memory)

In [ ]:
# Create in-memory vector store for demo
print(" Creating in-memory vector store...")
vector_store = VectorStore(use_pgvector=False)

# Add chunks with embeddings (preserves chunk IDs!)
start_time = time.time()
stored_chunk_ids = vector_store.add_chunks(chunks, embeddings)
add_time = time.time() - start_time

print(f"\n Chunks stored successfully:")
print(f"   Stored {len(stored_chunk_ids)} chunks")
print(f"   Time: {add_time*1000:.2f}ms")
print(f"   Chunk IDs preserved: {stored_chunk_ids}")

## Step 5: Retrieve Similar Chunks - Query 1

In [ ]:
# Query 1: Refund policy
query_1 = "What is the refund policy?"
print(f" Query: {query_1}")

# Generate query embedding
query_embedding = embed_query(query_1, model)
print(f"   Query embedding dimension: {query_embedding.shape[0]}")

# Search
start_time = time.time()
results_1 = vector_store.search(query_embedding, top_k=3)
search_time = time.time() - start_time

print(f"\n Retrieved {len(results_1)} chunks ({search_time*1000:.2f}ms):")
for i, result in enumerate(results_1, 1):
    print(f"\n   Result {i}:")
    print(f"   - Chunk ID: {result['chunk_id']}")
    print(f"   - Document: {result['document_id']}")
    print(f"   - Page: {result['metadata']['page_number']}")
    print(f"   - Source: {result['metadata']['source']}")
    print(f"   - Similarity: {result['score']:.4f}")
    print(f"   - Text: {result['chunk_text'][:120]}...")

## Step 6: Generate Citations from Retrieved Results

In [ ]:
# Extract unique citations
citations = []
seen_sources = set()

for result in results_1:
    chunk_id = result['chunk_id']
    file_name = result['metadata']['source']
    page_number = result['metadata']['page_number']
    
    key = (file_name, page_number, chunk_id)
    if key not in seen_sources:
        citations.append({
            "file_name": file_name,
            "page_number": page_number,
            "chunk_id": chunk_id
        })
        seen_sources.add(key)

print(f" Citations for Query 1:")
for i, citation in enumerate(citations, 1):
    print(f"   [{i}] {citation['file_name']} (page {citation['page_number']})")
    print(f"       Chunk: {citation['chunk_id']}")

## Step 7: Test Metadata Filtering

In [ ]:
# Query 2: Filter by page number
query_2 = "What is the warranty?"
print(f" Query: {query_2}")
print(f"   With filter: page_number = 3\n")

# Generate embedding
query_embedding_2 = embed_query(query_2, model)

# Search with filter
results_2 = vector_store.search(
    query_embedding_2,
    top_k=5,
    filter_metadata={"page_number": 3}
)

print(f" Retrieved {len(results_2)} chunks (filtered):")
for i, result in enumerate(results_2, 1):
    print(f"\n   Result {i}:")
    print(f"   - Chunk ID: {result['chunk_id']}")
    print(f"   - Page: {result['metadata']['page_number']}")
    print(f"   - Similarity: {result['score']:.4f}")
    print(f"   - Text: {result['chunk_text'][:120]}...")

## Step 8: Generate Simple Answer from Context

In [ ]:
# Generate prompt
prompt = build_rag_prompt(query_1, results_1)
print(f" Generated RAG Prompt:")
print("="*60)
print(prompt)
print("="*60)

# Generate simple answer (no LLM yet)
simple_answer = generate_simple_answer(query_1, results_1)
print(f"\n Simple Answer:")
print(f"   {simple_answer}")

## Step 9: Show RAG Response Contract

In [ ]:
# Build complete response matching RAG response contract
rag_response = {
    "question": query_1,
    "answer": None,  # No LLM yet
    "retrieved_context": [
        {
            "chunk_id": r['chunk_id'],
            "document_id": r['document_id'],
            "file_name": r['metadata']['source'],
            "page_number": r['metadata']['page_number'],
            "similarity_score": round(r['score'], 4),
            "chunk_text": r['chunk_text']
        }
        for r in results_1
    ],
    "citations": citations,
    "status": "retrieval_only",
    "model": "all-MiniLM-L6-v2",
    "metadata": {
        "retrieval_time_ms": round(search_time * 1000, 2),
        "num_chunks_searched": len(chunks),
        "top_k": 3,
        "filter_applied": None
    }
}

print(f" RAG Response (JSON):")
print(json.dumps(rag_response, indent=2))

## Step 10: Retrieval Evaluation

In [ ]:
# Simulate retrieval evaluation test cases
test_queries = [
    {
        "query": "What is the refund policy?",
        "expected_pages": [1],
        "expected_topic": "refund"
    },
    {
        "query": "How do I return items?",
        "expected_pages": [2],
        "expected_topic": "returns"
    },
    {
        "query": "What is the warranty coverage?",
        "expected_pages": [3],
        "expected_topic": "warranty"
    }
]

print(f" Retrieval Evaluation Results:\n")
print(f"{'Query':<40} {'Top Page':<12} {'Score':<8} {'Hit?':<6}")
print("="*70)

hits_at_1 = 0
total_score = 0

for test in test_queries:
    query_emb = embed_query(test["query"], model)
    results = vector_store.search(query_emb, top_k=1)
    
    if results:
        top_result = results[0]
        retrieved_page = top_result['metadata']['page_number']
        score = top_result['score']
        is_hit = retrieved_page in test['expected_pages']
        
        if is_hit:
            hits_at_1 += 1
        total_score += score
        
        print(f"{test['query']:<40} {retrieved_page:<12} {score:.4f}    {' YES' if is_hit else ' NO':<6}")

print("="*70)
print(f"\n Metrics:")
print(f"   Hit@1: {hits_at_1}/{len(test_queries)} ({100*hits_at_1/len(test_queries):.1f}%)")
print(f"   Avg Similarity Score: {total_score/len(test_queries):.4f}")

## Summary: Week 3 RAG Pipeline Demo Complete 

### Deliverables Demonstrated:

1.  **Document Loading**: Loaded sample documents simulating Duy's PDF extraction
2.  **Chunking with Metadata**: Created chunks with page numbers and source tracking
3.  **Embeddings**: Generated 384-dimensional embeddings using all-MiniLM-L6-v2
4.  **Vector Storage**: Stored chunks with preserved chunk IDs in vector store
5.  **Semantic Search**: Retrieved top-3 chunks by similarity
6.  **Chunk Preservation**: Verified chunk_id format (doc_001_page_1_chunk_000)
7.  **Citations**: Generated source citations with file names and page numbers
8.  **Metadata Filtering**: Filtered retrieval by page_number
9.  **RAG Response**: Built response matching RAG response contract
10.  **Evaluation**: Completed retrieval evaluation with Hit@1 metric

### Next Steps (Week 4):

- Implement pgvector backend with PostgreSQL
- Add LLM answer generation
- Scale testing with larger document sets
- Performance optimization
- Integration with Phi's backend and Hung's chatbot UI